
# 02 — Landing to Consumption

**Purpose:** Read raw data from **landing**, apply **data quality checks and cleansing**, and write a **Delta table** to the **consumption** Volume.

| Item | Value |
|------|--------|
| Input | Landing Parquet (5 months) |
| Output | Delta at `/Volumes/.../consumption/yellow_taxi` |
| Columns | 5 required fields (case specification) |

This notebook is the main **PySpark transformation** step.

---


## Configuration and imports

- Read landing **one month at a time** to avoid Parquet schema conflicts across files
- Union months with `unionByName`

---

In [0]:
# --- Config ---
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, year, count, sum as spark_sum
CATALOG = "workspace"
SCHEMA = "nyc_taxi"
months = ["01", "02", "03", "04", "05"]
landing_base = f"/Volumes/{CATALOG}/{SCHEMA}/landing/yellow_taxi/2023"
consumption_path = f"/Volumes/{CATALOG}/{SCHEMA}/consumption/yellow_taxi"

## Read landing layer

Defines `read_month()` to:

1. Load one Parquet file
2. Select only the **5 required columns**
3. Cast numeric fields; keep timestamps as read from Parquet

Appends each month to `dfs`, then unions into `df_landing`.

---

In [0]:
# --- Read landing (one month at a time — avoids Parquet schema conflicts across files) ---
def read_month(path: str) -> DataFrame:
    return (
        spark.read.parquet(path)
        .select(
            col("VendorID").cast("long"),
            col("passenger_count").cast("long"),
            col("total_amount").cast("double"),
            col("tpep_pickup_datetime"),
            col("tpep_dropoff_datetime"),
        )
    )
dfs = []
for m in months:
    path = f"{landing_base}/{m}/yellow_tripdata_2023-{m}.parquet"
    print(f"Reading {path}")
    df = read_month(path)
    print(f"  rows: {df.count():,}")
    dfs.append(df)
df_landing = reduce(DataFrame.unionByName, dfs)
print(f"Total landing (union): {df_landing.count():,}")


## Data quality checks (before cleansing)

Aggregated metrics to understand data issues before filters:

- Null counts per required column
- Rows outside year 2023
- Negative `total_amount` / `passenger_count`
- Invalid trip duration (`dropoff <= pickup`)

Use results to tune cleansing rules or document exclusions in the README.

---

In [0]:
# --- Data quality checks (before cleansing) ---
display(df_landing.limit(5))
df_landing.printSchema()

checks_before = df_landing.agg(
    count("*").alias("total_rows"),
    spark_sum(col("VendorID").isNull().cast("int")).alias("null_vendor"),
    spark_sum(col("passenger_count").isNull().cast("int")).alias("null_passenger"),
    spark_sum(col("total_amount").isNull().cast("int")).alias("null_amount"),
    spark_sum(col("tpep_pickup_datetime").isNull().cast("int")).alias("null_pickup"),
    spark_sum(col("tpep_dropoff_datetime").isNull().cast("int")).alias("null_dropoff"),
    spark_sum((year(col("tpep_pickup_datetime")) != 2023).cast("int")).alias("pickup_not_2023"),
    spark_sum((col("total_amount") < 0).cast("int")).alias("negative_amount"),
    spark_sum((col("passenger_count") < 0).cast("int")).alias("negative_passenger"),
    spark_sum(
        (col("tpep_dropoff_datetime") <= col("tpep_pickup_datetime")).cast("int")
    ).alias("invalid_trip_duration"),
)

display(checks_before)

## Cleansing and business rules

Builds `df_treated` with filters aligned to the case:

| Rule | Rationale |
|------|-----------|
| Non-null required columns | Case data contract |
| Pickup in Jan–May 2023 | Case period |
| `total_amount >= 0` | Invalid fare removal |
| `0 <= passenger_count <= 9` | Plausible yellow taxi capacity |
| `dropoff > pickup` | Valid trip timeline |

Final step applies explicit **casts** so all months share the same schema before write.

---

In [0]:
# --- Cleansing / business rules (case: Jan–May 2023, 5 required columns) ---
df_treated = (
    df_landing
    .filter(col("VendorID").isNotNull())
    .filter(col("passenger_count").isNotNull())
    .filter(col("total_amount").isNotNull())
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    # Case period: January through May 2023
    .filter(col("tpep_pickup_datetime") >= "2023-01-01")
    .filter(col("tpep_pickup_datetime") < "2023-06-01")
    .filter(year(col("tpep_pickup_datetime")) == 2023)
    .filter(col("total_amount") >= 0)
    .filter(col("passenger_count") >= 0)
    .filter(col("passenger_count") <= 9)
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
)

print(f"Before: {df_landing.count():,}")
print(f"After:  {df_treated.count():,}")
print(f"Removed: {df_landing.count() - df_treated.count():,}")

## Post-cleansing validation

- Row count before vs after
- Distribution by pickup year (expect **2023** only)
- Sample of key columns

---

In [0]:
# --- Post-cleansing checks ---
display(
    df_treated.groupBy(year(col("tpep_pickup_datetime")).alias("pickup_year"))
    .count()
    .orderBy("pickup_year")
)

display(df_treated.select("tpep_pickup_datetime", "total_amount", "passenger_count").limit(10))

## Data Schema Applying

Normalyzing data for load.

---

In [0]:
df_treated = (
    df_treated
    # ... filters ...
    .select(
        col("VendorID").cast("long"),
        col("passenger_count").cast("long"),
        col("total_amount").cast("double"),
        col("tpep_pickup_datetime").cast("timestamp"),
        col("tpep_dropoff_datetime").cast("timestamp"),
    )
)



## Write consumption layer (Delta on Volume)

Writes `df_treated` as a **Delta Lake** dataset on the consumption Volume.

| Option | When to use |
|--------|-------------|
| `overwriteSchema=true` | Schema changed during development |
| `dbutils.fs.rm(consumption_path, recurse=True)` | Alternative full reset before write |

**Enterprise (illustrative):** same code with `consumption_path = "s3://my-company-datalake/curated/nyc_taxi/yellow_taxi"`.

---

In [0]:
# --- Load consumption layer (Delta files on Volume) ---
(
    df_treated.write
    .format("delta")
    .mode("overwrite")
    .save(consumption_path)
)

print(f"Delta written to: {consumption_path}")
display(dbutils.fs.ls(consumption_path))

## (Illustrative) Enterprise — write consumption to S3

```python
# consumption_path = "s3://my-company-datalake/curated/nyc_taxi/yellow_taxi"
# df_treated.write.format("delta").mode("overwrite").save(consumption_path)
```

Not used on Free Edition; documented for production comparison.

---



## Smoke test — read back Delta

Confirms the consumption path is readable and row count matches expectations.

---

In [0]:
# --- Smoke test: read back the Delta dataset ---
df_consumption_files = spark.read.format("delta").load(consumption_path)
print(f"Rows in consumption: {df_consumption_files.count():,}")
display(df_consumption_files.limit(5))

## Consumption layer complete

Next step: run **`03_register_table`** to expose the dataset as `nyc_taxi.yellow_trips_consumption` for SQL consumers.

---